# Alerting & Resource Monitor Coverage Audit Notebook

Checks whether the account's monitoring is actually wired to *do* something — a resource
monitor with a threshold but no suspend action, or an alert defined but never firing, provides
much less protection than it looks like on paper. Run top to bottom. It surfaces:

1. **Resource monitor inventory** — every monitor, its quota, and its configured actions
2. **Warehouses with no resource monitor** — uncapped spend risk
3. **Monitor thresholds vs. actual spend** — quotas that are unrealistic (already exceeded, or so high they'd never trigger)
4. **Alert inventory & firing history** — defined alerts, whether they're actually running, and whether their action executes successfully
5. **Notification integration coverage** — whether alerts/monitors/tasks can actually deliver a notification anywhere
6. **Task failure notification coverage** — whether failing tasks (from the pipeline health notebook) are wired to notify anyone
7. **A rolled-up coverage summary**

### Prerequisites
- A role with `IMPORTED PRIVILEGES` on `SNOWFLAKE` (or `ACCOUNTADMIN`), plus visibility into
  resource monitors and alerts (`ACCOUNTADMIN` or explicit grants).
- Nothing here is destructive. Every cell reads.


In [ ]:
-- ============================================================
-- PARAMETERS
-- ============================================================
SET lookback_days = 30;

SELECT $lookback_days AS lookback_days;


## 1. Resource monitor inventory

A monitor with only a `NOTIFY` action still lets spend run away if nobody reads the
notification in time — the ones with a `SUSPEND` or `SUSPEND_IMMEDIATE` action at a
meaningful threshold are the ones actually preventing runaway cost, not just reporting it
after the fact.


In [ ]:
-- All resource monitors and their configuration
SHOW RESOURCE MONITORS;


In [ ]:
-- Parse for quota, current usage, and configured triggers/actions
SELECT
    "name" AS monitor_name,
    "credit_quota",
    "used_credits",
    "remaining_credits",
    "level",
    "frequency",
    "start_time",
    "end_time",
    "suspend_at",
    "suspend_immediate_at",
    "notify_at",
    "owner"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY ("suspend_at" IS NULL AND "suspend_immediate_at" IS NULL) DESC, monitor_name;


## 2. Warehouses with no resource monitor

Every uncovered warehouse is unbounded spend risk — nothing stops it from running (and
billing) indefinitely if a query or a stuck task keeps it alive.


In [ ]:
-- Current warehouses (for cross-referencing against monitor assignment)
SHOW WAREHOUSES;


In [ ]:
-- Warehouses and their assigned resource monitor, if any
SELECT
    "name" AS warehouse_name,
    "size",
    "resource_monitor",
    "auto_suspend",
    "owner"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY ("resource_monitor" IS NULL OR "resource_monitor" = '') DESC, warehouse_name;


Rows where `resource_monitor` is blank have no warehouse-level cap at all — check
whether an account-level monitor covers them instead (Section 1's `level = 'ACCOUNT'` rows) before
treating every blank row as a gap.


## 3. Monitor thresholds vs. actual spend

A quota that's already been exceeded (and the monitor is just sitting past its limit with no
suspend action) or one set so high it's essentially decorative are both common findings —
this compares configured quotas against real recent credit usage per warehouse.


In [ ]:
-- Recent actual credit usage by warehouse, for comparison against monitor quotas above
SELECT
    warehouse_name,
    ROUND(SUM(credits_used), 1) AS credits_used_in_window,
    ROUND(SUM(credits_used) / $lookback_days, 2) AS avg_daily_credits
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY credits_used_in_window DESC;


Compare `avg_daily_credits` here against each monitor's `credit_quota` and `frequency`
(daily/weekly/monthly) from Section 1 — a monthly quota smaller than 30x the average daily
spend is already being exceeded most months; a quota many times larger than realistic spend
is unlikely to ever trigger and isn't really protecting anything.


## 4. Alert inventory & firing history

An alert that's defined but suspended, or whose condition never evaluates true, provides no
actual protection — this checks both the definition and whether it has fired recently.


In [ ]:
-- All alerts and their current state
SHOW ALERTS IN ACCOUNT;


In [ ]:
-- Alert firing/execution history
SELECT
    database_name,
    schema_name,
    name AS alert_name,
    state,
    LEFT(condition_text, 200) AS condition_preview,
    LEFT(action_text, 200)    AS action_preview,
    scheduled_time,
    completed_time,
    ROUND(DATEDIFF('second', scheduled_time, completed_time), 1) AS duration_sec
FROM snowflake.account_usage.alert_history
WHERE scheduled_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
ORDER BY scheduled_time DESC
LIMIT 300;


In [ ]:
-- Alerts defined but with zero execution history in the window (suspended, or never firing)
WITH fired AS (
    SELECT DISTINCT database_name, schema_name, name
    FROM snowflake.account_usage.alert_history
    WHERE scheduled_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
)
SELECT
    "database_name",
    "schema_name",
    "name" AS alert_name,
    "state",
    "condition",
    "schedule"
FROM TABLE(RESULT_SCAN((SELECT LAST_QUERY_ID(-2))))  -- reuses SHOW ALERTS from Section 4; re-run that SHOW directly above if this errors
WHERE ("database_name", "schema_name", "name") NOT IN (
        SELECT database_name, schema_name, name FROM fired
      )
ORDER BY "state" DESC, alert_name;


## 5. Notification integration coverage

Resource monitors, alerts, and tasks can all be configured to notify — but only if a
notification integration actually exists for them to send through. No integration means
`NOTIFY` actions everywhere are silently doing nothing.


In [ ]:
-- Notification integrations defined
SHOW NOTIFICATION INTEGRATIONS;


If Section 5 returns nothing, every `NOTIFY`/`NOTIFY_USERS` action configured anywhere in
Sections 1 and 4 has no delivery mechanism — that's a single fix (create and wire up a
notification integration, typically to email or a cloud messaging service) that would make
every existing monitor and alert in the account actually useful, rather than needing to touch
each one individually.


## 6. Task failure notification coverage

Cross-references against the pipeline & task health notebook: do tasks that actually fail have
an `ERROR_INTEGRATION` configured so someone gets notified, or does a failure just sit in
`TASK_HISTORY` until someone happens to look?


In [ ]:
-- Tasks and their configured error notification integration, if any
SHOW TASKS IN ACCOUNT;


In [ ]:
SELECT
    "database_name",
    "schema_name",
    "name" AS task_name,
    "state",
    "error_integration",
    "warehouse"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "state" = 'started'
  AND ("error_integration" IS NULL OR "error_integration" = '')
ORDER BY "database_name", "schema_name", task_name;


Cross-reference this list against the pipeline & task health notebook's Section 2 (task
failure rates) — a task with both a non-trivial failure rate *and* no `error_integration`
configured is failing without anyone being told.


## 7. Rolled-up coverage summary

| Area | Finding | Risk if unaddressed |
|---|---|---|
| Warehouses without a resource monitor | *Section 2* | Uncapped compute spend |
| Monitors with no suspend action | *Section 1* | Spend visibility without spend control |
| Monitor quotas unrealistic vs. actual usage | *Section 3* | False sense of protection either direction |
| Alerts defined but never firing | *Section 4* | Silent monitoring gap |
| No notification integration configured | *Section 5* | Every NOTIFY action account-wide is non-functional |
| Failing tasks with no error notification | *Section 6* | Pipeline failures go unnoticed |


### Notes, caveats, and next steps

- **Section 5 is the highest-leverage single fix in this notebook.** If no notification
  integration exists, that's one piece of setup that unblocks every other notify-based control
  in the account at once, rather than a per-monitor or per-alert fix.
- **A `SUSPEND_IMMEDIATE` monitor is a blunt instrument.** It stops runaway spend but also
  stops legitimate in-flight work — appropriate for genuinely uncapped-risk warehouses, less so
  for one running a business-critical pipeline where a graceful `NOTIFY` + manual response
  might be the better tradeoff. Worth a deliberate choice per warehouse rather than one setting
  applied everywhere.
- **Pair directly with the pipeline & task health notebook and the warehouse right-sizing
  notebook** — this one asks "would we find out if something went wrong," those ask "did
  something go wrong," and together they're a more complete operational picture than either
  alone.
